In [0]:
'''

Notebook name: bv_data_ingestion_notebook
Author: piyush
Date: 8th September 2026
Description: 
            1) Using external location connecting to gen2 lake raw folder
            2) create a dataframe on raw folder and audit columns and overwrite the bronze
            3) apply the transformation from bronze to silver
            4) move files from raw folder to archieve folder
            

'''

'\n\nNotebook name: bv_data_ingestion_notebook\nAuthor: aheli\nDate: 8th September 2026\nDescription: \n            1) Using external location connecting to gen2 lake raw folder\n            2) create a dataframe on raw folder and audit columns and overwrite the bronze\n            3) apply the transformation from bronze to silver\n            4) move files from raw folder to archieve folder\n            \n\n'

In [0]:
# table_nm = 'reviews'

table_nm = dbutils.widgets.get('tablename')

In [0]:
##Using external location connecting to gen2 lake raw folder

extrnl_loc = "abfss://practice-container@apstorageproject.dfs.core.windows.net/"
raw_path = extrnl_loc + "raw/"
bv_raw_path = extrnl_loc + "raw/BV/"
tbl_bv_raw_path = extrnl_loc + "raw/BV/" + table_nm

catalog_nm = 'ap_proj_catalog'

bronze_schema  = 'bronze'
bronze_tbl = table_nm + "_stg"

silver_schema = 'silver'
silver_tbl = table_nm

archive_path = extrnl_loc + "archive/"
bv_archive_path = extrnl_loc + "archive/BV/"
tbl_bv_archive_path = extrnl_loc + "archive/BV/" + table_nm

In [0]:
##create a dataframe on raw folder and audit columns and overwrite the bronze

In [0]:
df = spark.read.format("json").load(tbl_bv_raw_path)
df.display()

cust_id,prod_id,rating,review_id,review_text,reviewdate
C1,PROD001,5,REVW001,product is not good,2026-09-02
C2,PROD002,4,REVW002,Nice product,2026-09-02
C3,PROD003,3,REVW003,worthable product,2026-09-02
C4,PROD006,2,REVW004,use less product,2026-09-02
C1,PROD006,1,REVW005,product is not good,2026-09-02
C2,PROD001,5,REVW006,Nice product,2026-09-02


In [0]:
from datetime import datetime

dt = datetime.now().strftime("%Y-%m-%d")

dt

'2026-09-09'

In [0]:
from pyspark.sql.functions import *

In [0]:
df = df.withColumn("loadDate", lit(dt))

df.display()

cust_id,prod_id,rating,review_id,review_text,reviewdate,loadDate
C1,PROD001,5,REVW001,product is not good,2026-09-02,2026-09-09
C2,PROD002,4,REVW002,Nice product,2026-09-02,2026-09-09
C3,PROD003,3,REVW003,worthable product,2026-09-02,2026-09-09
C4,PROD006,2,REVW004,use less product,2026-09-02,2026-09-09
C1,PROD006,1,REVW005,product is not good,2026-09-02,2026-09-09
C2,PROD001,5,REVW006,Nice product,2026-09-02,2026-09-09


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(f'{catalog_nm}.{bronze_schema}.{bronze_tbl}')

In [0]:
df = spark.sql(f'select * from {catalog_nm}.{bronze_schema}.{bronze_tbl}')

display(df)

cust_id,prod_id,rating,review_id,review_text,reviewdate,loadDate
C1,PROD001,5,REVW001,product is not good,2026-09-02,2026-09-09
C2,PROD002,4,REVW002,Nice product,2026-09-02,2026-09-09
C3,PROD003,3,REVW003,worthable product,2026-09-02,2026-09-09
C4,PROD006,2,REVW004,use less product,2026-09-02,2026-09-09
C1,PROD006,1,REVW005,product is not good,2026-09-02,2026-09-09
C2,PROD001,5,REVW006,Nice product,2026-09-02,2026-09-09


In [0]:
if table_nm == 'reviews':
    df.write.format("delta").mode("append").saveAsTable(f"{catalog_nm}.{silver_schema}.{silver_tbl}")
elif table_nm == 'categories':
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog_nm}.{silver_schema}.{silver_tbl}")

In [0]:
df = spark.sql(f'select * from {catalog_nm}.{silver_schema}.{silver_tbl}')

display(df)

cust_id,prod_id,rating,review_id,review_text,reviewdate,loadDate
C1,PROD001,5,REVW001,product is not good,2026-09-02,2026-09-09
C2,PROD002,4,REVW002,Nice product,2026-09-02,2026-09-09
C3,PROD003,3,REVW003,worthable product,2026-09-02,2026-09-09
C4,PROD006,2,REVW004,use less product,2026-09-02,2026-09-09
C1,PROD006,1,REVW005,product is not good,2026-09-02,2026-09-09
C2,PROD001,5,REVW006,Nice product,2026-09-02,2026-09-09


In [0]:
dbutils.fs.mkdirs(tbl_bv_archive_path)


In [0]:
file_lst = dbutils.fs.ls(tbl_bv_raw_path)

file_lst

[FileInfo(path='abfss://practice-container@apstorageproject.dfs.core.windows.net/raw/BV/reviews/reviews_02092026.json', name='reviews_02092026.json', size=782, modificationTime=1788703419000)]

In [0]:
for file in file_lst:
    dbutils.fs.mv(file.path, tbl_bv_archive_path)